# SBAIL Archive — JSON Content Tests

This notebook checks the JSON files produced for SBAIL Archive are correct. It reads from `aria_s_bails.create_sbail_json_content` (column `JSON_content`) and verifies the values against silver and bronze sources independently — so it catches bugs the pipeline itself wouldn't notice.

Each section below runs one test. Results land in a CSV plus `test_automation_runs2` / `test_automation_results2` / `test_automation_perfresults2` so they show up in the dashboards. Every result row carries a plain-English `description` explaining what was checked.

Per-segment mappings (WHEN-maps, concat formulas, pass-through fields) were extracted from the dev pipeline code as a regression baseline — see `hmcts_work/upgrades/archive_json_tests/source_map_sbail.md` for the audit.

In [0]:
########################
# test Setup
########################

state_under_test = "archive_sbail_json"
run_tag_default  = "sbail_json_content"

dbutils.widgets.text("run_tag", run_tag_default)
dbutils.widgets.text("hive_schema", "aria_s_bails")
dbutils.widgets.text("sample_size", "20")

run_tag     = dbutils.widgets.get("run_tag") or run_tag_default
hive_schema = dbutils.widgets.get("hive_schema") or "aria_s_bails"
sample_size = int(dbutils.widgets.get("sample_size") or 20)

# Segment shape (regression baseline from source_map_sbail.md — change in the generator, not here)
GOLD_TABLE        = "create_sbail_json_content"
JSON_CONTENT_COL  = "JSONContent"   # column name in gold table holding the JSON string
KEY_COL_IN_GOLD   = "CaseNo"
KEY_IN_JSON       = "CaseNo"
# Where case-level scalars live in the JSON. APPEALS/JOH/TD use [] (root). BAIL/SBAIL use ['Case_detail', 0].
SCALAR_ROOT_PATH  = ['Case_detail', 0]
# Bronze CaseNos sometimes have a /YYYY suffix that gold/silver strip. Set to "/\d{4}$" to strip on bronze side
# before joining to gold's short CaseNo. Empty string means no transform.
BRONZE_KEY_STRIP_REGEX = '/\\d{4}$'
REQUIRED_SCALAR_KEYS = ['CaseNo']
EXPECTED_ARRAY_KEYS  = ['Case_detail', 'all_status_objects', 'm5_history_details', 'financial_condition_details', 'bfdiary_details', 'linked_files_details', 'appeal_category_details', 'linked_cases_aggregated']
ARRAY_TO_SILVER      = {'all_status_objects': 'silver_sbail_m7_status', 'm5_history_details': 'silver_sbail_m5_history', 'bfdiary_details': 'silver_sbail_m4_bf_diary', 'linked_files_details': 'silver_sbail_m6_link', 'appeal_category_details': 'silver_sbail_m8'}
ARRAY_TO_BRONZE      = {'all_status_objects': 'bronze_sbail_status_sc_ra_cs', 'm5_history_details': 'bronze_sbail_ac_history_users', 'bfdiary_details': 'bronze_sbail_ac_bfdiary_bftype', 'linked_files_details': 'bronze_sbail_ac_link_linkdetail', 'appeal_category_details': 'bronze_sbail_ac_appealcategory_category'}
ARRAY_PRIMARY_KEYS   = {'m5_history_details': ('HistoryId',), 'all_status_objects': ('StatusId',)}
# Arrays excluded from the count-based tests (Test 4 cardinality, Test 9 bronze row counts).
# These are link-group / relational concepts where the source is not row-per-case — counting
# rows where CaseNo = case under-counts because the pipeline resolves them via a join column
# like LinkNo. The array is still validated by presence/absence (Test 2/3) and value-range
# / deep-equality tests where applicable.
COUNT_TEST_EXCLUDES  = ['linked_files_details']

def _get_scalar(obj, field):
    """Return obj[SCALAR_ROOT_PATH...][field], traversing list indices and dict keys."""
    if obj is None:
        return None
    cur = obj
    for k in SCALAR_ROOT_PATH:
        if cur is None:
            return None
        try:
            cur = cur[k]
        except (KeyError, IndexError, TypeError):
            return None
    if not isinstance(cur, dict):
        return None
    return cur.get(field)

# Test 5 (primary appellant filter)
HAS_APPELLANT_PARTITION = True
M2_SILVER_TABLE         = 'silver_sbail_m2_case_appellant'

# Tests 6, 7, 10 (WHEN-maps, concat)
WHEN_MAP_RANGES        = {'$.Case_detail[0].BailTypeDesc': {'', None, 'Scottish Bail', 'Other', 'Bail'}, '$.Case_detail[0].CourtPreferenceDesc': {'All Male,', 'All Female,', None, ''}, '$.Case_detail[0].InterpreterDesc': {'', None, 'Yes', 'No', 'Unknown'}, '$.Case_detail[0].TypeOfCostAwardDesc': {'', None, 'Fee Costs', 'Unreasonable Behaviour', 'Wasted Costs', 'Unknown', 'General Costs'}, '$.Case_detail[0].ApplyingPartyDesc': {'', None, 'Respondent', 'Appellant', 'Unknown'}, '$.Case_detail[0].PayingPartyDesc': {'Respondent', 'Respondent Rep', None, '', 'Interested Party', 'Appellant Rep', 'Appellant', 'Surety/cautioner', 'Unknown'}, '$.Case_detail[0].CostsAwardDecisionDesc': {'', None, 'Interim field', 'Refused', 'Unknown', 'Granted'}, '$.Case_detail[0].AppealCategoriesDesc': {'NO', '', None, 'YES'}, '$.Case_detail[0].AppellantDetainedDesc': {'', None, 'IRC', 'Other', 'No', 'Unknown', 'HMP'}}
WHEN_MAP_ROUNDTRIP     = [{'bronze_table': 'bronze_sbail_ac_cr_cs_ca_fl_cres_mr_res_lang', 'bronze_col': 'Interpreter', 'json_field': 'InterpreterDesc', 'mapping': {1: 'Yes', 2: 'No'}, 'default': 'Unknown'}, {'bronze_table': 'bronze_sbail_ac_cr_cs_ca_fl_cres_mr_res_lang', 'bronze_col': 'AppealCategories', 'json_field': 'AppealCategoriesDesc', 'mapping': {1: 'YES'}, 'default': 'NO'}, {'bronze_table': 'bronze_sbail_ac_cr_cs_ca_fl_cres_mr_res_lang', 'bronze_col': 'CourtPreference', 'json_field': 'CourtPreferenceDesc', 'mapping': {1: 'All Male,', 2: 'All Female,'}, 'default': ''}, {'bronze_table': 'bronze_sbail_ac_ca_apt_country_detc', 'bronze_col': 'AppellantDetained', 'json_field': 'AppellantDetainedDesc', 'mapping': {1: 'HMP', 2: 'IRC', 3: 'No', 4: 'Other'}, 'default': 'Unknown'}]
CONCAT_FIELDS          = [{'json_field': 'FileLocation', 'silver_table': 'silver_sbail_m1_case_details', 'bronze_table': 'bronze_sbail_ac_cr_cs_ca_fl_cres_mr_res_lang', 'parts': ['FileLocationCentre', 'FileLocationDepartment', 'FileLocationNote'], 'sep': ', '}]

# Test 8 (bronze pass-through)
BRONZE_PASSTHROUGH     = [{'bronze_table': 'bronze_sbail_ac_cr_cs_ca_fl_cres_mr_res_lang', 'scalar_fields': ['HORef', 'AppealCaseNote', 'CountryId', 'PortId', 'NationalityId'], 'date_fields': ['DateOfIssue', 'DateServed', 'DateReceived']}, {'bronze_table': 'bronze_sbail_ac_ca_apt_country_detc', 'scalar_fields': ['AppellantName', 'AppellantForenames', 'AppellantTitle', 'AppellantPostcode', 'PortReference'], 'date_fields': ['AppellantBirthDate']}]


In [0]:
import os
import sys
import json as _json
import time as perf_time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import col

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/Archive_" + state_under_test.replace("archive_", "").replace("_json", "") + "_Tests"
os.makedirs(test_results_path, exist_ok=True)

for _mod in list(sys.modules.keys()):
    if _mod.startswith("Test_Functions") or _mod == "models.test_result":
        del sys.modules[_mod]

from models.test_result import TestResult
from Test_Functions.report_helpers import (
    build_report_folder,
    write_csv, write_xlsx, write_html,
    count_by_status,
    create_run, create_results, create_perf_results,
)

print(f"User: {run_user}")
print(f"state_under_test: {state_under_test}")
print(f"hive_schema: {hive_schema}  gold: {GOLD_TABLE}  key: {KEY_COL_IN_GOLD}/{KEY_IN_JSON}")
print(f"sample_size: {sample_size}")
print(f"Results folder root: {test_results_path}")

In [0]:
#######################
# Spark Config
#######################
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
first_row = config.first()
env = first_row["env"].strip().lower()
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"

client_secret = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-TENANT-ID")
client_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-ID")

for storage in (f"ingest{lz_key}curated{env}", f"ingest{lz_key}xcutting{env}",
                f"ingest{lz_key}raw{env}", f"ingest{lz_key}landing{env}", f"ingest{lz_key}external{env}"):
    spark.conf.set(f"fs.azure.account.auth.type.{storage}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print(f"env={env}  lz_key={lz_key}")

## Load the gold, silver, and bronze tables

The cell below loads:
- **create_sbail_json_content** — the table that contains the `JSON_content` column we're checking.
- Each silver source table needed for Tests 4 and 11.
- Each bronze source table needed for Tests 7, 8, 9, 10.
- Optional silver M2 table for Test 5 (primary-appellant filter).

All loads are best-effort — if a table doesn't exist in this env, the dependent test reports `NO_DATA` rather than crash.

In [0]:
def _try_load(table_name):
    try:
        return spark.table(f"hive_metastore.{hive_schema}.{table_name}")
    except Exception as _e:
        print(f"{table_name} not loadable: {_e}")
        return None

gold_json = _try_load(GOLD_TABLE)
if gold_json is not None:
    gold_json = gold_json.cache()    # every test reads gold; cache pays for itself many times over
    gold_count = gold_json.count()
    print(f"{GOLD_TABLE}: {gold_count} rows (cached)")
else:
    gold_count = 0
    print(f"{GOLD_TABLE}: not loaded")

SILVER_BY_ARRAY = {}
for json_key, silver_table_name in ARRAY_TO_SILVER.items():
    SILVER_BY_ARRAY[json_key] = _try_load(silver_table_name)

BRONZE_BY_ARRAY = {}
for json_key, bronze_table_name in ARRAY_TO_BRONZE.items():
    BRONZE_BY_ARRAY[json_key] = _try_load(bronze_table_name)

silver_m2 = _try_load(M2_SILVER_TABLE) if M2_SILVER_TABLE else None

# Bronze tables for Tests 7, 8, 10 (pre-loaded for speed)
BRONZE_BY_NAME = {}
for rt in WHEN_MAP_ROUNDTRIP:
    name = rt["bronze_table"]
    if name not in BRONZE_BY_NAME:
        BRONZE_BY_NAME[name] = _try_load(name)
for cf in CONCAT_FIELDS:
    name = cf["bronze_table"]
    if name not in BRONZE_BY_NAME:
        BRONZE_BY_NAME[name] = _try_load(name)
for bp in BRONZE_PASSTHROUGH:
    name = bp["bronze_table"]
    if name not in BRONZE_BY_NAME:
        BRONZE_BY_NAME[name] = _try_load(name)

## Pick a representative sample of cases

Tests 4-11 only check a sample of cases (cheaper than running over the whole table). Picked randomly with a fixed seed so reruns are reproducible.

In [0]:
run_start_datetime = datetime.utcnow()
all_test_results = []
perf_timings = []
t0 = perf_time.time()

def _run_timed(label, fn, *args, **kwargs):
    before = len(all_test_results)
    _t0 = perf_time.time()
    try:
        return fn(*args, **kwargs)
    finally:
        elapsed = perf_time.time() - _t0
        added = len(all_test_results) - before
        perf_timings.append({"test_name": label, "elapsed_seconds": elapsed, "result_count": added})
        print(f"  [{elapsed:6.2f}s, +{added:4d} results] {label}")

def pick_sample(df, n, key_col):
    if df is None: return []
    try:
        return [getattr(r, key_col) for r in df.select(key_col).orderBy(F.rand(seed=42)).limit(n).collect()]
    except Exception as e:
        print(f"sample pick failed: {e}")
        return []

# Stringify SAMPLE so int-keyed gold cols (e.g. JOH AdjudicatorId) work alongside string-keyed (CaseNo).
# Every downstream comparison column is .cast("string") to match; every dict keyed by case is str(key).
SAMPLE = [str(x) for x in pick_sample(gold_json, sample_size, KEY_COL_IN_GOLD)]
print(f"Sample size {len(SAMPLE)}: {SAMPLE[:5]}{'...' if len(SAMPLE) > 5 else ''}")

# Pre-parse JSON for the sampled rows
SAMPLE_JSON = {}
if gold_json is not None and SAMPLE:
    for r in gold_json.filter(col(KEY_COL_IN_GOLD).cast("string").isin(*SAMPLE)).select(KEY_COL_IN_GOLD, JSON_CONTENT_COL).collect():
        key = str(getattr(r, KEY_COL_IN_GOLD))
        try:
            SAMPLE_JSON[key] = _json.loads(getattr(r, JSON_CONTENT_COL)) if getattr(r, JSON_CONTENT_COL) else None
        except Exception as _e:
            SAMPLE_JSON[key] = None
            print(f"sample JSON parse failed for {key}: {_e}")
print(f"Pre-parsed {sum(1 for v in SAMPLE_JSON.values() if v is not None)} sample JSONs")

## Test 1 — Are the JSON files well-formed?

The cell below loops over every row in the gold table and checks three things:
- the `JSON_content` string actually parses as JSON;
- the top-level `CaseNo` inside the JSON matches the row's `CaseNo` column;
- no row's `JSON_content` starts with `'failure'` (pipeline error markers that should have been dropped before gold).

In [0]:
def test_json_parseable_and_key_matches(gold_df):
    if gold_df is None:
        all_test_results.append(TestResult(test_name="test_json_parseable_and_key_matches",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="NO_DATA", message="gold table not loaded"))
        return
    try:
        parsed = gold_df.select(
            col(KEY_COL_IN_GOLD).alias("row_key"),
            F.get_json_object(col(JSON_CONTENT_COL), f"$.{KEY_IN_JSON}").alias("json_key"),
            col(JSON_CONTENT_COL).isNull().alias("is_null_content"),
        )
        null_content = parsed.filter(col("is_null_content")).count()
        unparseable  = parsed.filter(~col("is_null_content") & col("json_key").isNull()).count()
        mismatched   = parsed.filter(col("json_key").isNotNull() & (col("json_key") != col("row_key"))).count()
        for field, n in [("null_JSON_content", null_content),
                         ("unparseable_or_missing_key", unparseable),
                         (f"{KEY_IN_JSON}_mismatch_row_vs_json", mismatched)]:
            status = "PASS" if n == 0 else "FAIL"
            msg = "" if n == 0 else f"{n} rows"
            all_test_results.append(TestResult(test_name="test_json_parseable_and_key_matches",
                                               test_field=field, test_from_state=state_under_test,
                                               status=status, message=msg))
    except Exception as e:
        all_test_results.append(TestResult(test_name="test_json_parseable_and_key_matches",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="ERROR", message=f"{type(e).__name__}: {e}"))

def test_no_failure_json_content(gold_df):
    if gold_df is None:
        all_test_results.append(TestResult(test_name="test_no_failure_json_content",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="NO_DATA", message="gold table not loaded"))
        return
    try:
        n = gold_df.filter(F.lower(col(JSON_CONTENT_COL)).like("failure%")).count()
        status = "PASS" if n == 0 else "FAIL"
        msg = "" if n == 0 else f"{n} rows have failure% JSON"
        all_test_results.append(TestResult(test_name="test_no_failure_json_content",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status=status, message=msg))
    except Exception as e:
        all_test_results.append(TestResult(test_name="test_no_failure_json_content",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_json_parseable_and_key_matches", test_json_parseable_and_key_matches, gold_json)
_run_timed("test_no_failure_json_content", test_no_failure_json_content, gold_json)

## Test 2 — Are the required top-level keys present?

Splits into two checks:
- **Required scalars** — must be on every row. A missing one is a FAIL.
- **Expected array keys** — may be absent on rows where the case genuinely has no source data; reported with a count, not a FAIL.

In [0]:
def test_required_top_level_keys(gold_df):
    if gold_df is None:
        all_test_results.append(TestResult(test_name="test_required_scalar_keys",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="NO_DATA", message="gold table not loaded"))
        return
    total = gold_df.count()
    for k in REQUIRED_SCALAR_KEYS:
        try:
            missing_df = gold_df.filter(F.get_json_object(col(JSON_CONTENT_COL), f"$.{k}").isNull())
            missing = missing_df.count()
            if missing == 0:
                all_test_results.append(TestResult(test_name="test_required_scalar_keys",
                                                   test_field=k, test_from_state=state_under_test,
                                                   status="PASS", message=""))
            else:
                examples = [getattr(r, KEY_COL_IN_GOLD) for r in missing_df.select(KEY_COL_IN_GOLD).limit(5).collect()]
                all_test_results.append(TestResult(test_name="test_required_scalar_keys",
                                                   test_field=k, test_from_state=state_under_test,
                                                   status="FAIL",
                                                   message=f"{missing} rows missing key; examples: {examples}"))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_required_scalar_keys",
                                               test_field=k, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

    for k in EXPECTED_ARRAY_KEYS:
        try:
            missing = gold_df.filter(F.get_json_object(col(JSON_CONTENT_COL), f"$.{k}").isNull()).count()
            present = total - missing
            all_test_results.append(TestResult(test_name="test_expected_array_keys",
                                               test_field=k, test_from_state=state_under_test,
                                               status="PASS",
                                               message=f"present on {present}/{total} rows; absent={missing}"))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_expected_array_keys",
                                               test_field=k, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_required_top_level_keys", test_required_top_level_keys, gold_json)

## Test 3 — Are missing array keys actually empty in silver?

For each absent array key (from Test 2), the cell below looks up the case in the corresponding silver source. If silver has zero rows for that case the absence is expected (PASS). If silver has rows that the JSON dropped, that's a real bug (FAIL).

In [0]:
def test_array_absences_match_silver(gold_df):
    if gold_df is None:
        all_test_results.append(TestResult(test_name="test_array_absences_match_silver",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="NO_DATA", message="gold table not loaded"))
        return
    # Single pass over gold: compute absence flag per array key in one select, then cache.
    keys_to_check = [k for k, v in SILVER_BY_ARRAY.items() if v is not None]
    if keys_to_check:
        flag_cols = [F.get_json_object(col(JSON_CONTENT_COL), f"$.{k}").isNull().alias(f"absent__{k}") for k in keys_to_check]
        absent_flags = gold_df.select(col(KEY_COL_IN_GOLD), *flag_cols).cache()
        absent_flags.count()
    else:
        absent_flags = None

    # One distinct-CaseNo scan per UNIQUE silver_df, cached by id so shared silvers scan once.
    silver_cases_by_id = {}

    for json_key, silver_df in SILVER_BY_ARRAY.items():
        if silver_df is None:
            all_test_results.append(TestResult(test_name="test_array_absences_match_silver",
                                               test_field=json_key, test_from_state=state_under_test,
                                               status="NO_DATA", message="silver source not loadable"))
            continue
        try:
            absent_cases = [getattr(r, KEY_COL_IN_GOLD) for r in
                            absent_flags.filter(col(f"absent__{json_key}"))
                                        .select(KEY_COL_IN_GOLD).collect()]
            if not absent_cases:
                all_test_results.append(TestResult(test_name="test_array_absences_match_silver",
                                                   test_field=json_key, test_from_state=state_under_test,
                                                   status="PASS", message="key present on every row"))
                continue
            sid = id(silver_df)
            if sid not in silver_cases_by_id:
                silver_key_col = "CaseNo" if "CaseNo" in silver_df.columns else KEY_COL_IN_GOLD
                # Single distinct scan, return small set of all CaseNos that have any silver row.
                silver_cases_by_id[sid] = {r[silver_key_col] for r in silver_df.select(silver_key_col).distinct().collect()}
            silver_cases = silver_cases_by_id[sid]
            # Intersect in Python — no giant isin OR-expression.
            real_loss = [c for c in absent_cases if c in silver_cases]
            if not real_loss:
                msg = f"all {len(absent_cases)} absences confirmed empty in silver"
                status = "PASS"
            else:
                msg = f"{len(real_loss)}/{len(absent_cases)} absences have silver rows (real data loss); examples: {real_loss[:5]}"
                status = "FAIL"
            all_test_results.append(TestResult(test_name="test_array_absences_match_silver",
                                               test_field=json_key, test_from_state=state_under_test,
                                               status=status, message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_array_absences_match_silver",
                                               test_field=json_key, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_array_absences_match_silver", test_array_absences_match_silver, gold_json)

## Test 4 — Do JSON arrays match silver counts?

For each sampled case, the cell below counts silver rows independently (per case, our own filter) and compares that count to the length of the matching JSON array. Catches the pipeline silently dropping or duplicating rows.

In [0]:
def test_array_cardinality_vs_silver():
    if not SAMPLE:
        all_test_results.append(TestResult(test_name="test_array_cardinality_vs_silver",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="PASS", message="not applicable: no sample available (gold may be empty for this segment)"))
        return
    silver_counts = {}
    for json_key, silver_df in SILVER_BY_ARRAY.items():
        if json_key in COUNT_TEST_EXCLUDES:
            continue
        if silver_df is None:
            silver_counts[json_key] = None
            continue
        try:
            silver_key_col = "CaseNo" if "CaseNo" in silver_df.columns else KEY_COL_IN_GOLD
            cnt = silver_df.filter(col(silver_key_col).cast("string").isin(*SAMPLE)).groupBy(silver_key_col).count().collect()
            silver_counts[json_key] = {str(getattr(r, silver_key_col)): r["count"] for r in cnt}
        except Exception as e:
            silver_counts[json_key] = None
            print(f"silver count for {json_key} failed: {e}")

    for case_key in SAMPLE:
        obj = SAMPLE_JSON.get(case_key)
        if obj is None: continue
        for json_key in SILVER_BY_ARRAY.keys():
            if json_key in COUNT_TEST_EXCLUDES:
                continue
            sc = silver_counts.get(json_key)
            if sc is None:
                all_test_results.append(TestResult(test_name="test_array_cardinality_vs_silver",
                                                   test_field=f"{case_key}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status="NO_DATA", message="silver source not loadable"))
                continue
            arr = obj.get(json_key) or []
            json_n = len(arr) if isinstance(arr, list) else -1
            silver_n = sc.get(case_key, 0)
            ok = json_n == silver_n
            all_test_results.append(TestResult(test_name="test_array_cardinality_vs_silver",
                                               test_field=f"{case_key}:{json_key}",
                                               test_from_state=state_under_test,
                                               status=("PASS" if ok else "FAIL"),
                                               message=("" if ok else f"json={json_n} silver={silver_n}")))

_run_timed("test_array_cardinality_vs_silver", test_array_cardinality_vs_silver)

## Test 5 — Primary appellant filter

The pipeline filters silver_*_m2 (case appellant) to `Relationship IS NULL`, so only primary appellants make it into the JSON. The cell below checks, for sampled cases, that the AppellantId (or equivalent person identifier) shown in the JSON is one that silver_*_m2 considers primary. Dependents should NOT appear at the root.

In [0]:
def test_primary_appellant_filter():
    if silver_m2 is None:
        all_test_results.append(TestResult(test_name="test_primary_appellant_filter",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="NO_DATA", message="silver_m2 not loadable"))
        return
    if not SAMPLE:
        all_test_results.append(TestResult(test_name="test_primary_appellant_filter",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="PASS", message="not applicable: no sample available (gold may be empty for this segment)"))
        return
    # silver_*_m2 already filters Relationship IS NULL. So every row in it is a primary appellant for its case.
    # For each sampled case, confirm the JSON's appellant identity matches one of silver_m2's rows.
    try:
        rows = silver_m2.filter(col("CaseNo").cast("string").isin(*SAMPLE)).select("CaseNo", "AppellantId").collect()
        m2_by_case = {}
        for r in rows:
            m2_by_case.setdefault(r.CaseNo, set()).add(r.AppellantId)
    except Exception as e:
        all_test_results.append(TestResult(test_name="test_primary_appellant_filter",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="ERROR", message=f"silver_m2 read failed: {e}"))
        return

    passes, fails, examples = 0, 0, []
    for case_no in SAMPLE:
        obj = SAMPLE_JSON.get(case_no)
        if obj is None: continue
        # The JSON's appellant identity might be under multiple field names. Try several.
        json_app_id = _get_scalar(obj, "AppellantId") or _get_scalar(obj, "AppellantID")
        primary_ids = m2_by_case.get(case_no, set())
        # If neither side has data, treat as NO_DATA per-case (skip).
        if json_app_id is None and not primary_ids:
            continue
        ok = (json_app_id in primary_ids) if primary_ids else False
        if ok:
            passes += 1
        else:
            fails += 1
            if len(examples) < 3:
                examples.append(f"{case_no}: json_app_id={json_app_id} primary_set={list(primary_ids)[:3]}")

    if passes == 0 and fails == 0:
        status, msg = "NO_DATA", "no sampled cases had a comparable appellant id"
    elif fails == 0:
        status, msg = "PASS", f"verified {passes} cases"
    else:
        status, msg = "FAIL", f"pass={passes} fail={fails}; " + "; ".join(examples)
    all_test_results.append(TestResult(test_name="test_primary_appellant_filter",
                                       test_field="SAMPLE", test_from_state=state_under_test,
                                       status=status, message=msg))

_run_timed("test_primary_appellant_filter", test_primary_appellant_filter)

## Test 6 — Are coded values translated to the right labels?

Some fields in the source data are numeric codes that silver translates to text labels. The cell below collects every distinct JSON value at each WHEN-mapped path across the gold table and checks it's in the documented set of allowed labels (locked in from the dev pipeline as a regression baseline).

In [0]:
def test_when_map_value_ranges(gold_df):
    if gold_df is None:
        all_test_results.append(TestResult(test_name="test_when_map_value_ranges",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="NO_DATA", message="gold table not loaded"))
        return
    for path, allowed in WHEN_MAP_RANGES.items():
        if allowed is None:
            all_test_results.append(TestResult(test_name="test_when_map_value_ranges",
                                               test_field=path, test_from_state=state_under_test,
                                               status="NO_DATA",
                                               message="field has open-ended pass-through fallback; range check not meaningful"))
            continue
        try:
            distinct_rows = gold_df.select(F.get_json_object(col(JSON_CONTENT_COL), path).alias("v")).distinct().collect()
            offenders = sorted({(r.v if r.v is not None else "__NULL__") for r in distinct_rows if r.v not in allowed})
            status = "PASS" if not offenders else "FAIL"
            msg = "" if not offenders else f"unexpected values: {offenders[:10]}"
            all_test_results.append(TestResult(test_name="test_when_map_value_ranges",
                                               test_field=path, test_from_state=state_under_test,
                                               status=status, message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_when_map_value_ranges",
                                               test_field=path, test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_when_map_value_ranges", test_when_map_value_ranges, gold_json)

## Test 7 — Re-do the code→label translation ourselves and check

Two parts:
- **WHEN-map round-trip:** read the raw bronze code, apply our own Python mapping (the test IS the spec; defined at the top of the notebook in `WHEN_MAP_ROUNDTRIP`), check it matches the JSON value for sampled cases.
- **Concat round-trip from silver:** rebuild named fields (like `FileLocation`) from silver components and check the result matches the JSON.

In [0]:
def test_when_map_roundtrip():
    if not SAMPLE:
        all_test_results.append(TestResult(test_name="test_when_map_roundtrip",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="PASS", message="not applicable: no sample available (gold may be empty for this segment)"))
        return
    for rt in WHEN_MAP_ROUNDTRIP:
        bronze_df = BRONZE_BY_NAME.get(rt["bronze_table"])
        if bronze_df is None or rt["bronze_col"] not in bronze_df.columns:
            all_test_results.append(TestResult(test_name="test_when_map_roundtrip",
                                               test_field=rt["json_field"], test_from_state=state_under_test,
                                               status="NO_DATA",
                                               message=f"bronze table or column not loadable: {rt['bronze_table']}.{rt['bronze_col']}"))
            continue
        try:
            bronze_key_col_orig = "CaseNo" if "CaseNo" in bronze_df.columns else KEY_COL_IN_GOLD
            # Cast to string first (some bronze keys like AdjudicatorId are INT but SAMPLE is string),
            # then trim trailing whitespace, then apply optional year-suffix strip.
            _key_expr = F.trim(col(bronze_key_col_orig).cast("string"))
            if BRONZE_KEY_STRIP_REGEX:
                _key_expr = F.regexp_replace(_key_expr, BRONZE_KEY_STRIP_REGEX, "")
            bronze_df = bronze_df.withColumn("_match_key", _key_expr)
            bronze_key_col = "_match_key"
            rows = bronze_df.filter(col(bronze_key_col).isin(*SAMPLE)).select(bronze_key_col, rt["bronze_col"]).collect()
            raw_by_case = {getattr(r, bronze_key_col): getattr(r, rt["bronze_col"]) for r in rows}
            passes, fails, examples = 0, 0, []
            for case_no in SAMPLE:
                obj = SAMPLE_JSON.get(case_no)
                if obj is None: continue
                raw_v = raw_by_case.get(case_no)
                if raw_v is None: continue
                expected = rt["mapping"].get(raw_v, rt["default"])
                actual = _get_scalar(obj, rt["json_field"])
                if (expected or "") == (actual or ""):
                    passes += 1
                else:
                    fails += 1
                    if len(examples) < 3:
                        examples.append(f"{case_no}: raw={raw_v} expected={expected!r} actual={actual!r}")
            if passes == 0 and fails == 0:
                status, msg = "NO_DATA", "no sample case had this bronze value"
            elif fails == 0:
                status, msg = "PASS", f"pass={passes}"
            else:
                status, msg = "FAIL", f"pass={passes} fail={fails}; " + "; ".join(examples)
            all_test_results.append(TestResult(test_name="test_when_map_roundtrip",
                                               test_field=rt["json_field"], test_from_state=state_under_test,
                                               status=status, message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_when_map_roundtrip",
                                               test_field=rt["json_field"], test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

def test_concat_field_roundtrip():
    if not SAMPLE:
        all_test_results.append(TestResult(test_name="test_concat_field_roundtrip",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="PASS", message="not applicable: no sample available (gold may be empty for this segment)"))
        return
    for cf in CONCAT_FIELDS:
        # Silver-sourced concat round-trip
        silver_df = None
        try:
            silver_df = spark.table(f"hive_metastore.{hive_schema}.{cf['silver_table']}")
        except Exception:
            pass
        if silver_df is None:
            all_test_results.append(TestResult(test_name="test_concat_field_roundtrip",
                                               test_field=cf["json_field"], test_from_state=state_under_test,
                                               status="NO_DATA", message=f"silver table not loadable: {cf['silver_table']}"))
            continue
        missing_cols = [c for c in cf["parts"] if c not in silver_df.columns]
        if missing_cols:
            all_test_results.append(TestResult(test_name="test_concat_field_roundtrip",
                                               test_field=cf["json_field"], test_from_state=state_under_test,
                                               status="NO_DATA", message=f"silver missing columns: {missing_cols}"))
            continue
        try:
            silver_key_col = "CaseNo" if "CaseNo" in silver_df.columns else KEY_COL_IN_GOLD
            rows = silver_df.filter(col(silver_key_col).cast("string").isin(*SAMPLE)).select(silver_key_col, *cf["parts"]).collect()
            by_case = {str(getattr(r, silver_key_col)): r for r in rows}
            passes, fails, examples = 0, 0, []
            for case_no in SAMPLE:
                obj = SAMPLE_JSON.get(case_no)
                row = by_case.get(case_no)
                if obj is None or row is None: continue
                parts = [str(getattr(row, p)) for p in cf["parts"] if getattr(row, p)]
                expected = cf["sep"].join(parts)
                actual = _get_scalar(obj, cf["json_field"])
                if (expected or "") == (actual or ""):
                    passes += 1
                else:
                    fails += 1
                    if len(examples) < 3:
                        examples.append(f"{case_no}: expected={expected!r} actual={actual!r}")
            if passes == 0 and fails == 0:
                status, msg = "NO_DATA", "no sample case had silver components"
            elif fails == 0:
                status, msg = "PASS", f"pass={passes}"
            else:
                status, msg = "FAIL", f"pass={passes} fail={fails}; " + "; ".join(examples)
            all_test_results.append(TestResult(test_name="test_concat_field_roundtrip",
                                               test_field=cf["json_field"], test_from_state=state_under_test,
                                               status=status, message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_concat_field_roundtrip",
                                               test_field=cf["json_field"], test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_when_map_roundtrip", test_when_map_roundtrip)
_run_timed("test_concat_field_roundtrip", test_concat_field_roundtrip)

## Test 8 — Do pass-through fields come straight through from bronze?

For sampled cases, compares the bronze value for each pass-through field directly to the JSON value. Catches silent corruption between bronze and gold for fields silver doesn't transform.

Dates are compared on the `YYYY-MM-DD` prefix (timezone-tolerant). Numerics use a float-equal fallback.

In [0]:
def _values_match(a, b, is_date=False):
    if a is None and b is None: return True
    if a is None or b is None:
        if a == "" or b == "": return True
        return False
    sa, sb = str(a).strip(), str(b).strip()
    if sa == sb: return True
    if is_date and len(sa) >= 10 and len(sb) >= 10 and sa[:10] == sb[:10]:
        return True
    try:
        if float(sa) == float(sb): return True
    except (ValueError, TypeError):
        pass
    return False

def test_bronze_passthrough_scalars():
    if not SAMPLE:
        all_test_results.append(TestResult(test_name="test_bronze_passthrough_scalars",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="PASS", message="not applicable: no sample available (gold may be empty for this segment)"))
        return
    for bp in BRONZE_PASSTHROUGH:
        bronze_df = BRONZE_BY_NAME.get(bp["bronze_table"])
        if bronze_df is None:
            for f in bp.get("scalar_fields", []) + bp.get("date_fields", []):
                all_test_results.append(TestResult(test_name="test_bronze_passthrough_scalars",
                                                   test_field=f, test_from_state=state_under_test,
                                                   status="NO_DATA", message=f"bronze not loadable: {bp['bronze_table']}"))
            continue
        try:
            bronze_key_col_orig = "CaseNo" if "CaseNo" in bronze_df.columns else KEY_COL_IN_GOLD
            # Cast to string first (some bronze keys like AdjudicatorId are INT but SAMPLE is string),
            # then trim trailing whitespace, then apply optional year-suffix strip.
            _key_expr = F.trim(col(bronze_key_col_orig).cast("string"))
            if BRONZE_KEY_STRIP_REGEX:
                _key_expr = F.regexp_replace(_key_expr, BRONZE_KEY_STRIP_REGEX, "")
            bronze_df = bronze_df.withColumn("_match_key", _key_expr)
            bronze_key_col = "_match_key"
            all_fields = bp.get("scalar_fields", []) + bp.get("date_fields", [])
            available = [f for f in all_fields if f in bronze_df.columns]
            if not available: continue
            rows = bronze_df.filter(col(bronze_key_col).isin(*SAMPLE)).select(bronze_key_col, *available).collect()
            by_case = {getattr(r, bronze_key_col): r.asDict() for r in rows}
            for field in available:
                is_date = field in bp.get("date_fields", [])
                passes, fails, examples = 0, 0, []
                for case_no in SAMPLE:
                    obj = SAMPLE_JSON.get(case_no)
                    row = by_case.get(case_no)
                    if obj is None or row is None: continue
                    bv = row.get(field)
                    jv = _get_scalar(obj, field)
                    if _values_match(bv, jv, is_date=is_date):
                        passes += 1
                    else:
                        fails += 1
                        if len(examples) < 3:
                            examples.append(f"{case_no}: bronze={bv!r} json={jv!r}")
                if passes == 0 and fails == 0:
                    status, msg = "NO_DATA", "no rows compared"
                elif fails == 0:
                    status, msg = "PASS", f"pass={passes}"
                else:
                    status, msg = "FAIL", f"pass={passes} fail={fails}; " + "; ".join(examples)
                all_test_results.append(TestResult(test_name="test_bronze_passthrough_scalars",
                                                   test_field=field, test_from_state=state_under_test,
                                                   status=status, message=msg))
        except Exception as e:
            for f in bp.get("scalar_fields", []) + bp.get("date_fields", []):
                all_test_results.append(TestResult(test_name="test_bronze_passthrough_scalars",
                                                   test_field=f, test_from_state=state_under_test,
                                                   status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_bronze_passthrough_scalars", test_bronze_passthrough_scalars)

## Test 9 — Do JSON arrays match bronze counts?

Same idea as Test 4 (silver count vs JSON length) but using bronze. Catches data loss between bronze and silver — a bug Test 4 would miss because if silver lost a row both its count AND the JSON's length would shrink in lockstep.

In [0]:
def test_bronze_array_row_counts():
    if not SAMPLE:
        all_test_results.append(TestResult(test_name="test_bronze_array_row_counts",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="PASS", message="not applicable: no sample available (gold may be empty for this segment)"))
        return
    bronze_counts = {}
    for json_key, bronze_df in BRONZE_BY_ARRAY.items():
        if json_key in COUNT_TEST_EXCLUDES:
            continue
        if bronze_df is None:
            bronze_counts[json_key] = None
            continue
        try:
            bronze_key_col_orig = "CaseNo" if "CaseNo" in bronze_df.columns else KEY_COL_IN_GOLD
            # Cast to string first (some bronze keys like AdjudicatorId are INT but SAMPLE is string),
            # then trim trailing whitespace, then apply optional year-suffix strip.
            _key_expr = F.trim(col(bronze_key_col_orig).cast("string"))
            if BRONZE_KEY_STRIP_REGEX:
                _key_expr = F.regexp_replace(_key_expr, BRONZE_KEY_STRIP_REGEX, "")
            bronze_df = bronze_df.withColumn("_match_key", _key_expr)
            bronze_key_col = "_match_key"
            cnt = bronze_df.filter(col(bronze_key_col).isin(*SAMPLE)).groupBy(bronze_key_col).count().collect()
            bronze_counts[json_key] = {getattr(r, bronze_key_col): r["count"] for r in cnt}
        except Exception as e:
            bronze_counts[json_key] = None
            print(f"bronze count for {json_key} failed: {e}")

    for case_key in SAMPLE:
        obj = SAMPLE_JSON.get(case_key)
        if obj is None: continue
        for json_key in BRONZE_BY_ARRAY.keys():
            if json_key in COUNT_TEST_EXCLUDES:
                continue
            bc = bronze_counts.get(json_key)
            if bc is None:
                all_test_results.append(TestResult(test_name="test_bronze_array_row_counts",
                                                   test_field=f"{case_key}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status="NO_DATA", message="bronze source not loadable"))
                continue
            arr = obj.get(json_key) or []
            json_n = len(arr) if isinstance(arr, list) else -1
            bronze_n = bc.get(case_key, 0)
            ok = json_n == bronze_n
            all_test_results.append(TestResult(test_name="test_bronze_array_row_counts",
                                               test_field=f"{case_key}:{json_key}",
                                               test_from_state=state_under_test,
                                               status=("PASS" if ok else "FAIL"),
                                               message=("" if ok else f"json={json_n} bronze={bronze_n}")))

_run_timed("test_bronze_array_row_counts", test_bronze_array_row_counts)

## Test 10 — Re-do the concat fields from bronze and check

Same as Test 7's concat round-trip, but the cell below pulls components from bronze instead of silver. Useful diagnostic: if Test 7 (silver) passes but this test (bronze) fails, the bronze→silver step is the breaker.

In [0]:
def test_bronze_concat_roundtrip():
    if not SAMPLE:
        all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="PASS", message="not applicable: no sample available (gold may be empty for this segment)"))
        return
    for cf in CONCAT_FIELDS:
        bronze_df = BRONZE_BY_NAME.get(cf["bronze_table"])
        if bronze_df is None:
            all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                               test_field=cf["json_field"], test_from_state=state_under_test,
                                               status="NO_DATA", message=f"bronze not loadable: {cf['bronze_table']}"))
            continue
        missing_cols = [c for c in cf["parts"] if c not in bronze_df.columns]
        if missing_cols:
            all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                               test_field=cf["json_field"], test_from_state=state_under_test,
                                               status="NO_DATA", message=f"bronze missing columns: {missing_cols}"))
            continue
        try:
            bronze_key_col_orig = "CaseNo" if "CaseNo" in bronze_df.columns else KEY_COL_IN_GOLD
            # Cast to string first (some bronze keys like AdjudicatorId are INT but SAMPLE is string),
            # then trim trailing whitespace, then apply optional year-suffix strip.
            _key_expr = F.trim(col(bronze_key_col_orig).cast("string"))
            if BRONZE_KEY_STRIP_REGEX:
                _key_expr = F.regexp_replace(_key_expr, BRONZE_KEY_STRIP_REGEX, "")
            bronze_df = bronze_df.withColumn("_match_key", _key_expr)
            bronze_key_col = "_match_key"
            rows = bronze_df.filter(col(bronze_key_col).isin(*SAMPLE)).select(bronze_key_col, *cf["parts"]).collect()
            by_case = {getattr(r, bronze_key_col): r for r in rows}
            passes, fails, examples = 0, 0, []
            for case_no in SAMPLE:
                obj = SAMPLE_JSON.get(case_no)
                row = by_case.get(case_no)
                if obj is None or row is None: continue
                parts = [str(getattr(row, p)) for p in cf["parts"] if getattr(row, p)]
                expected = cf["sep"].join(parts)
                actual = _get_scalar(obj, cf["json_field"])
                if (expected or "") == (actual or ""):
                    passes += 1
                else:
                    fails += 1
                    if len(examples) < 3:
                        examples.append(f"{case_no}: expected={expected!r} actual={actual!r}")
            if passes == 0 and fails == 0:
                status, msg = "NO_DATA", "no sample case had bronze components"
            elif fails == 0:
                status, msg = "PASS", f"pass={passes}"
            else:
                status, msg = "FAIL", f"pass={passes} fail={fails}; " + "; ".join(examples)
            all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                               test_field=cf["json_field"], test_from_state=state_under_test,
                                               status=status, message=msg))
        except Exception as e:
            all_test_results.append(TestResult(test_name="test_bronze_concat_roundtrip",
                                               test_field=cf["json_field"], test_from_state=state_under_test,
                                               status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_bronze_concat_roundtrip", test_bronze_concat_roundtrip)

## Test 11 — Deep set-equality on key fields per case

For 5 deeply-sampled cases, the cell below pulls the SET of primary-key tuples from silver and the same set from the JSON array, then compares. Catches missing entries, extra entries, AND key-field corruption.

Only runs for arrays where a primary-key column is configured. Arrays without a known key report NO_DATA.

In [0]:
def test_sampled_deep_field_equality(n=5):
    deep_sample = SAMPLE[:n]
    if not deep_sample:
        all_test_results.append(TestResult(test_name="test_sampled_deep_field_equality",
                                           test_field="ALL", test_from_state=state_under_test,
                                           status="PASS", message="not applicable: no sample available (gold may be empty for this segment)"))
        return
    for case_key in deep_sample:
        obj = SAMPLE_JSON.get(case_key)
        if obj is None: continue
        for json_key, key_cols in ARRAY_PRIMARY_KEYS.items():
            silver_df = SILVER_BY_ARRAY.get(json_key)
            if silver_df is None:
                all_test_results.append(TestResult(test_name="test_sampled_deep_field_equality",
                                                   test_field=f"{case_key}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status="NO_DATA", message="silver source not loadable"))
                continue
            try:
                arr = obj.get(json_key) or []
                json_keys = set(tuple(d.get(k) for k in key_cols) for d in arr if isinstance(d, dict))
                silver_key_col = "CaseNo" if "CaseNo" in silver_df.columns else KEY_COL_IN_GOLD
                if not all(k in silver_df.columns for k in key_cols):
                    all_test_results.append(TestResult(test_name="test_sampled_deep_field_equality",
                                                       test_field=f"{case_key}:{json_key}",
                                                       test_from_state=state_under_test,
                                                       status="NO_DATA",
                                                       message=f"key columns {key_cols} not in silver"))
                    continue
                silver_rows = silver_df.filter(col(silver_key_col).cast("string") == case_key).select(*key_cols).collect()
                silver_keys = set(tuple(getattr(r, k, None) for k in key_cols) for r in silver_rows)
                missing = sorted(str(k) for k in (silver_keys - json_keys))
                extra   = sorted(str(k) for k in (json_keys - silver_keys))
                ok = not missing and not extra
                msg = "" if ok else f"missing={missing[:5]} extra={extra[:5]}"
                all_test_results.append(TestResult(test_name="test_sampled_deep_field_equality",
                                                   test_field=f"{case_key}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status=("PASS" if ok else "FAIL"), message=msg))
            except Exception as e:
                all_test_results.append(TestResult(test_name="test_sampled_deep_field_equality",
                                                   test_field=f"{case_key}:{json_key}",
                                                   test_from_state=state_under_test,
                                                   status="ERROR", message=f"{type(e).__name__}: {e}"))

_run_timed("test_sampled_deep_field_equality", test_sampled_deep_field_equality, 5)

In [0]:
elapsed = perf_time.time() - t0
counts = count_by_status(all_test_results)

print(f"Tests complete in {elapsed:.1f}s")
print(f"  Total : {counts['TOTAL']}")
print(f"  Pass  : {counts['PASS']}")
print(f"  Fail  : {counts['FAIL']}")
print(f"  NoData: {counts['NO_DATA']}")
print(f"  Error : {counts['ERROR']}")

DESCRIPTIONS = {'test_json_parseable_and_key_matches': "Test 1: Every gold row's JSON_content is valid JSON, and its top-level key matches the row's key column.", 'test_no_failure_json_content': "Test 1: No gold row has JSON_content starting with 'failure' (pipeline error markers that should have been dropped).", 'test_required_scalar_keys': 'Test 2: This required top-level scalar key is present (non-null) on every gold row.', 'test_expected_array_keys': 'Test 2: Reports how many rows have this array key present. Absence is normal when the case has zero source rows for the array.', 'test_array_absences_match_silver': 'Test 3: When the JSON omits this array key, silver has zero source rows for that case. A real bug is silver having rows that the JSON dropped.', 'test_array_cardinality_vs_silver': "Test 4: For this case, the JSON array length equals the count of silver rows for the array's source table.", 'test_primary_appellant_filter': 'Test 5: For sampled cases, the JSON appellant references the same person silver_*_m2 keeps as primary (Relationship IS NULL). Dependents should not appear at root.', 'test_when_map_value_ranges': 'Test 6: Every distinct JSON value at this path is in the documented set of mapped labels (regression baseline locked from dev code).', 'test_when_map_roundtrip': "Test 7: For sampled cases — raw bronze code, run through our own Python mapping, equals the JSON value. The test's map IS the contract; drift breaks the test.", 'test_concat_field_roundtrip': 'Test 7: Concat field rebuilt from silver components matches the JSON value (sampled cases).', 'test_bronze_passthrough_scalars': 'Test 8: Bronze value for this pass-through field equals the JSON value (sampled cases). Catches corruption between bronze and gold.', 'test_bronze_array_row_counts': 'Test 9: For this case, the JSON array length equals the count of bronze rows. Catches data loss between bronze and silver.', 'test_bronze_concat_roundtrip': 'Test 10: Concat field rebuilt from bronze components matches the JSON value (sampled cases).', 'test_sampled_deep_field_equality': 'Test 11: For this deeply-sampled case, the SET of primary-key tuples matches between silver and JSON.', '00_OVERALL_SUMMARY': 'Overall verdict for this run: total tests plus pass/fail/error/no-data counts.'}

for r in all_test_results:
    if not getattr(r, "description", ""):
        r.description = DESCRIPTIONS.get(r.test_name, "")

overall_status = "FAIL" if (counts["FAIL"] + counts["ERROR"]) > 0 else "PASS"
all_test_results.append(TestResult(
    test_name="00_OVERALL_SUMMARY",
    test_field="00_OVERALL",
    test_from_state=state_under_test,
    status=overall_status,
    message=(f"Total={counts['TOTAL']} Pass={counts['PASS']} "
             f"Fail={counts['FAIL']} Error={counts['ERROR']} "
             f"NoData={counts['NO_DATA']} Elapsed={elapsed:.1f}s"),
    description=DESCRIPTIONS.get("00_OVERALL_SUMMARY", ""),
))

## Save results

Headline row first; failures next; then everything else. Writes a CSV plus rows to `test_automation_runs2` / `test_automation_results2` / `test_automation_perfresults2`.

In [0]:
status_order = {"FAIL": 0, "ERROR": 1, "NO_DATA": 2, "PASS": 3}
def _sort_key(r):
    if str(getattr(r, "test_name", "")) == "00_OVERALL_SUMMARY":
        return (-1, "")
    return (status_order.get(str(r.status).upper(), 4), str(r.test_field))
all_test_results.sort(key=_sort_key)

folder, timestamp, _ = build_report_folder(test_results_path, f"{state_under_test}_{run_tag}")
print(f"Reports folder: {folder}")

try:
    csv_path  = write_csv(all_test_results, state_under_test, folder, timestamp)
    print(f"CSV  : {csv_path}")
except Exception as e:
    print(f"write_csv FAILED: {e}")

try:
    run_id = create_run(
        spark,
        run_user=run_user,
        run_start_datetime=run_start_datetime,
        run_by_automation_name="Archive_SBAIL_JSON_Tests",
        run_tag=run_tag,
        run_status=("FAILED" if any(str(getattr(r, "status", "")).upper().strip() in ("FAIL", "ERROR") for r in all_test_results) else "PASSED"),
        state_under_test=state_under_test,
    )
    print(f"Created run_id={run_id}")
    n = create_results(spark, run_id, all_test_results)
    print(f"Wrote {n} result rows")
    p = create_perf_results(spark, run_id, perf_timings, state_under_test)
    print(f"Wrote {p} perf rows")
except Exception as e:
    print(f"audit write FAILED: {e}")

## Final summary

Prints the overall PASS/FAIL verdict, totals, and first 10 failing rows (if any). Lists slowest tests too.

In [0]:
failures = counts["FAIL"] + counts["ERROR"]
verdict = "PASS" if failures == 0 else "FAIL"
banner = "=" * 70
print(banner)
print(f"  OVERALL RESULT: {verdict}    ({state_under_test})")
print(banner)
print(f"  Total tests : {counts['TOTAL']}")
print(f"  Passed      : {counts['PASS']}")
print(f"  Failed      : {counts['FAIL']}")
print(f"  Errored     : {counts['ERROR']}")
print(f"  No data     : {counts['NO_DATA']}")
print(f"  Elapsed     : {elapsed:.1f}s")
print(banner)
if verdict == "FAIL":
    print(f"  >>> {failures} failure(s) — see CSV / test_automation_results2 for detail.")
    fail_examples = [(r.test_name, r.test_field, r.message) for r in all_test_results
                     if str(getattr(r, "status", "")).upper().strip() in ("FAIL", "ERROR")][:10]
    for tn, tf, msg in fail_examples:
        print(f"    [{tn}] {tf} -- {msg[:160]}")
else:
    print(f"  >>> All {counts['TOTAL']} tests passed.")
print()
print("PERF: slowest tests")
print("-" * 70)
slowest = sorted(perf_timings, key=lambda p: p["elapsed_seconds"], reverse=True)[:5]
for p in slowest:
    print(f"  {p['elapsed_seconds']:7.2f}s  +{p['result_count']:4d} results  {p['test_name']}")
total_test_time = sum(p["elapsed_seconds"] for p in perf_timings)
print(f"  -------")
print(f"  {total_test_time:7.2f}s  total across {len(perf_timings)} test functions")